# Submission 11 — xlm-roberta-large + reuso de NB07 (BETO + XLM-R-base)

**Grupo 11 — Entrega 2 (Deep Learning) — subir el techo del modelo base**

## Hipótesis
El techo del pipeline lo pone el encoder base: el ensemble *base-size* de NB07 tope en
**0.3085 holdout / 0.30276 Kaggle**, y el híbrido de NB08 (legacy + DL) lo elevó a **0.31805**.
La palanca real es un encoder más fuerte. `xlm-roberta-large` (550M, hidden 1024, MIT) es la
**misma arquitectura** que el XLM-R-base que ya funciona — solo más capacidad y mejor
preentrenamiento multilingüe (cubre el latín/italiano/catalán del corpus). +0.02-0.04 esperado.

## Diseño (eficiente en GPU)
1. **Entrena SOLO el modelo grande** (~5h en T4). El BETO y el XLM-R-base ya están entrenados
   en NB07 → se **reutilizan** cargando su bundle (`modelo-final-dl` montado de input). Cero
   reentrenamiento de lo que ya existe.
2. **Split CANÓNICO de NB07** (`train_test_split`, `stratify`, `seed=42`): el holdout es
   exactamente el de NB07 → evaluar los modelos reusados ahí NO tiene fuga de datos.
3. **Ensemble** large + BETO (+ XLM-R-base) con pesos ∝ F1 en holdout. Bundle en formato
   uniforme → NB08 lo consume sin cambios y le suma el legacy (híbrido).

## Memoria en T4 (16GB)
xlm-roberta-large no cabe con la config base-size. Se usa **batch_size=8 + grad_accum=4**
(batch efectivo 32) + **gradient checkpointing** + fp16. Si hubiera OOM: bajar a batch_size=4,
grad_accum=8.

## Nota operativa Kaggle
Montar de input: `utils-py`, `dl-utils` (re-subido con el flag de checkpointing), la competencia,
y `modelo-final-dl` (bundle de NB07, para reusar BETO/XLM-R-base). El output (bundle + carpeta del
large) se publica como Kaggle Dataset (p.ej. `modelo-final-dl-large`) para alimentar NB08.


## 0. Setup

Sube `utils.py` y `dl_utils.py` como Kaggle Datasets (`utils-py`, `dl-utils`).


In [1]:
# %pip install -q transformers==4.45.* datasets accelerate safetensors sentencepiece scikit-learn joblib

import os, sys, warnings, json
warnings.filterwarnings("ignore")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

for p in [
    os.getcwd(),
    "/kaggle/input/datasets/diegoamolano/utils-py",
    "/kaggle/input/datasets/diegoamolano/dl-utils",
    "/kaggle/input/utils-py",
    "/kaggle/input/dl-utils",
]:
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import joblib
import utils, dl_utils
utils.set_global_seed(utils.SEED)

import torch
from transformers import AutoTokenizer
print("CUDA:", torch.cuda.is_available(),
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")


2026-05-21 06:35:23.935896: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779345324.153894      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779345324.223425      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779345324.765179      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779345324.765233      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779345324.765236      23 computation_placer.cc:177] computation placer alr

CUDA: True | device: Tesla T4


In [2]:
MODEL_NAME_LARGE = "FacebookAI/xlm-roberta-large"   # 550M, multilingüe, MIT
MAX_LENGTH = 384
ALPHA_CENTURY = 0.10

# Regularización (valores neutros validados en NB07/09)
DROPOUT = 0.1
USE_UNIFORM_SMOOTH = False
SMOOTH_EPS = 0.05
LOSS_KIND = "uniform" if USE_UNIFORM_SMOOTH else "ce"

# Config del modelo GRANDE en T4: batch chico + grad_accum + gradient checkpointing + fp16.
# lr más bajo (1e-5) que el base (2e-5) porque large es más sensible. epochs=6 con early
# stopping (patience 2): de-riesga el subentrenamiento (modo de fallo documentado en NB06)
# sin pasarse del límite de 12h (~50min/epoch con checkpointing → ~5h).
cfg_large = dl_utils.TrainConfig(
    max_length=MAX_LENGTH, batch_size=8, grad_accum=4, lr=1e-5, epochs=6,
    loss_kind=LOSS_KIND, smooth_eps=SMOOTH_EPS, alpha_century=ALPHA_CENTURY,
    precision="fp16", early_stopping_patience=2, gradient_checkpointing=True, log_every=50,
)

DATA_DIR = "../../data"
for cand in [
    "/kaggle/input/datasets/diegoamolano/parte-2-competencia-aprendizaje-de-maquina-2026-10",
    "/kaggle/input/aprendizaje-maquina-2026-10-parte-2",
    "../../data",
]:
    if os.path.exists(cand):
        DATA_DIR = cand
        break
OUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."

# Reuso de NB07: dónde está su bundle (modelo_final.joblib) + carpetas de modelos.
REUSE_NB07_MODELS = True
NB07_INPUT_DIRS = [
    "/kaggle/input/datasets/diegoamolano/modelo-final-dl",
    "/kaggle/input/modelo-final-dl",
    OUT_DIR, ".",
]
NB07_BUNDLE = next((os.path.join(d, "modelo_final.joblib") for d in NB07_INPUT_DIRS
                    if os.path.exists(os.path.join(d, "modelo_final.joblib"))), None)
print("DATA_DIR:", DATA_DIR, "| OUT_DIR:", OUT_DIR)
print("NB07 bundle:", NB07_BUNDLE, "| reuso:", REUSE_NB07_MODELS)
print(f"Regularización -> dropout={DROPOUT} | loss={LOSS_KIND} | lr={cfg_large.lr} | bs={cfg_large.batch_size}x{cfg_large.grad_accum} | ckpt={cfg_large.gradient_checkpointing}")


DATA_DIR: /kaggle/input/datasets/diegoamolano/parte-2-competencia-aprendizaje-de-maquina-2026-10 | OUT_DIR: /kaggle/working
NB07 bundle: /kaggle/input/datasets/diegoamolano/modelo-final-dl/modelo_final.joblib | reuso: True
Regularización -> dropout=0.1 | loss=ce | lr=1e-05 | bs=8x4 | ckpt=True


## 1. Datos + split CANÓNICO (idéntico a NB07 → holdout limpio para el reuso)

In [3]:
from sklearn.model_selection import train_test_split

corpus = utils.load_corpus(DATA_DIR)
utils.quick_summary(corpus)

# Split canónico: misma receta que NB07 (train_test_split sobre np.arange, stratify por
# década, seed=42). Garantiza que los modelos reusados de NB07 NO vieron este holdout.
texts = corpus.train["text_clean"].values
y_dec = np.array([utils.DECADE_TO_IDX[int(d)] for d in corpus.train["decade"].values])
y_cen = np.array([utils.CENTURY_TO_IDX[utils.decade_to_century(int(d))] for d in corpus.train["decade"].values])
idx_tr, idx_va = train_test_split(np.arange(len(corpus.train)), test_size=0.10,
                                  stratify=y_dec, random_state=utils.SEED)
val_texts = list(texts[idx_va])
eval_texts = corpus.eval_["text_clean"].tolist()
print("Train:", len(idx_tr), "| Val:", len(idx_va), "| Eval:", len(eval_texts))


Train shape: (31403, 3) | Eval shape: (3490, 3)
Décadas únicas: 39
Rango décadas: 150 - 188
Ejemplos por clase  min/mean/max: 754 / 805.2 / 848
Palabras por texto  median/p95/max: 50 / 255 / 1146
Train: 28262 | Val: 3141 | Eval: 3490


## 2. Modelo grande — xlm-roberta-large multi-task (fp16 + gradient checkpointing)

In [4]:
%%time
tok_large = AutoTokenizer.from_pretrained(MODEL_NAME_LARGE, use_fast=True)
model_large = dl_utils.MultiTaskClassifier(MODEL_NAME_LARGE, use_century=True, dropout=DROPOUT)
print("LARGE:", MODEL_NAME_LARGE, "|", round(sum(p.numel() for p in model_large.parameters())/1e6, 1), "M")

model_large, hist_large, f1_large_train = dl_utils.train_model(
    model_large, texts[idx_tr], y_dec[idx_tr], y_cen[idx_tr],
    texts[idx_va], y_dec[idx_va], y_cen[idx_va],
    tok_large, cfg=cfg_large, label="A-xlmr-large",
)
print("\nBest val F1 (train-time, truncado):", round(f1_large_train, 4))


config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: FacebookAI/xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LARGE: FacebookAI/xlm-roberta-large | 559.9 M
[A-xlmr-large] precisión=fp16 | steps/epoch=884 | total=5304 | warmup=530 | loss=ce | alpha_siglo=0.1
[A-xlmr-large] ep 1/6 step 50/5304 | loss 3.6021 | lr 9.43e-07 | gnorm 6.04 | 13 smp/s | ETA 215m51s
[A-xlmr-large] ep 1/6 step 100/5304 | loss 3.5786 | lr 1.89e-06 | gnorm 7.70 | 12 smp/s | ETA 222m05s
[A-xlmr-large] ep 1/6 step 150/5304 | loss 3.5116 | lr 2.83e-06 | gnorm 9.26 | 12 smp/s | ETA 222m22s
[A-xlmr-large] ep 1/6 step 200/5304 | loss 3.4531 | lr 3.77e-06 | gnorm 9.95 | 12 smp/s | ETA 221m27s
[A-xlmr-large] ep 1/6 step 250/5304 | loss 3.3960 | lr 4.72e-06 | gnorm 11.18 | 12 smp/s | ETA 220m12s
[A-xlmr-large] ep 1/6 step 300/5304 | loss 3.3367 | lr 5.66e-06 | gnorm 21.87 | 12 smp/s | ETA 218m32s
[A-xlmr-large] ep 1/6 step 350/5304 | loss 3.2746 | lr 6.60e-06 | gnorm 20.78 | 12 smp/s | ETA 216m44s
[A-xlmr-large] ep 1/6 step 400/5304 | loss 3.2272 | lr 7.55e-06 | gnorm 25.42 | 12 smp/s | ETA 214m58s
[A-xlmr-large] ep 1/6 step 450/53

## 3. Guardar large + proba holdout/eval (ventana deslizante) y liberar GPU

In [5]:
MODEL_LARGE_DIR = f"{OUT_DIR}/modelo_v11_A_xlmr_large"
dl_utils.save_multitask(model_large, tok_large, MODEL_LARGE_DIR)

# Proba con ventana deslizante (consistente con la inferencia de NB08).
pa_val = dl_utils.predict_logits_sliding(model_large, tok_large, val_texts, MAX_LENGTH, return_proba=True)
pa_eval = dl_utils.predict_logits_sliding(model_large, tok_large, eval_texts, MAX_LENGTH, return_proba=True)
f1_large = float(utils.compute_metrics(y_dec[idx_va], pa_val.argmax(1))["f1_macro"])
utils.print_metrics(utils.compute_metrics(y_dec[idx_va], pa_val.argmax(1)), "LARGE (holdout, sliding)")

del model_large
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Token indices sequence length is longer than the specified maximum sequence length for this model (531 > 512). Running this sequence through the model will result in indexing errors


[LARGE (holdout, sliding)] F1-macro         : 0.2941
[LARGE (holdout, sliding)] Accuracy         : 0.2980
[LARGE (holdout, sliding)] Mean dist (dec.) : 2.91
[LARGE (holdout, sliding)] <=1 decade       : 47.4%
[LARGE (holdout, sliding)] <=2 decades      : 63.5%
[LARGE (holdout, sliding)] Inter-century err: 21.6%


## 4. Reutilizar los modelos ya entrenados de NB07 (BETO + XLM-R-base)

Carga el bundle de NB07 y, para cada modelo, recalcula la proba en el holdout canónico y en el
eval. Sin reentrenamiento. Si el dataset de NB07 no está montado, el ensemble queda = large solo.


In [6]:
def _resolve_nb07_dir(d):
    if os.path.isdir(d):
        return d
    base = os.path.basename(d.rstrip("/"))
    for root in NB07_INPUT_DIRS:
        cand = os.path.join(root, base)
        if os.path.isdir(cand):
            return cand
    return None


reused = []
if REUSE_NB07_MODELS and NB07_BUNDLE and os.path.exists(NB07_BUNDLE):
    nb07 = joblib.load(NB07_BUNDLE)
    print("Reusando de NB07:", [m["name"] for m in nb07["models"]])
    for m in nb07["models"]:
        mdir = _resolve_nb07_dir(m["dir"])
        if mdir is None:
            print("  [omitido] no se encontró la carpeta:", m["dir"])
            continue
        tok = AutoTokenizer.from_pretrained(mdir, use_fast=True)
        mdl = dl_utils.load_multitask(mdir, use_century=m.get("use_century", True))
        pv = dl_utils.predict_logits_sliding(mdl, tok, val_texts, MAX_LENGTH, return_proba=True)
        pe = dl_utils.predict_logits_sliding(mdl, tok, eval_texts, MAX_LENGTH, return_proba=True)
        f1 = float(utils.compute_metrics(y_dec[idx_va], pv.argmax(1))["f1_macro"])
        reused.append({"name": m["name"], "dir": m["dir"], "use_century": m.get("use_century", True),
                       "proba_val": pv, "proba_eval": pe, "val_f1": f1})
        print(f"  reusado {m['name']}: F1 holdout {f1:.4f}")
        del mdl
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
else:
    print("Sin reuso de NB07 (toggle OFF o bundle ausente) -> ensemble = large solo")


Reusando de NB07: ['FacebookAI/xlm-roberta-base', 'dccuchile/bert-base-spanish-wwm-cased']


Token indices sequence length is longer than the specified maximum sequence length for this model (531 > 512). Running this sequence through the model will result in indexing errors


  reusado FacebookAI/xlm-roberta-base: F1 holdout 0.2825


Token indices sequence length is longer than the specified maximum sequence length for this model (568 > 512). Running this sequence through the model will result in indexing errors


  reusado dccuchile/bert-base-spanish-wwm-cased: F1 holdout 0.2908


## 5. Ensemble en holdout (pesos ∝ F1)

In [7]:
members = [{"name": MODEL_NAME_LARGE, "dir": MODEL_LARGE_DIR, "use_century": True,
            "proba_val": pa_val, "proba_eval": pa_eval, "val_f1": f1_large}] + reused

W = dl_utils.f1_weights([mb["val_f1"] for mb in members])
ens_val = dl_utils.combine_proba([mb["proba_val"] for mb in members], W)
val_metrics_ens = utils.compute_metrics(y_dec[idx_va], ens_val.argmax(1))

print("Miembros:", [mb["name"] for mb in members])
print("Pesos ∝F1:", [round(x, 3) for x in W])
for mb in members:
    utils.print_metrics(utils.compute_metrics(y_dec[idx_va], mb["proba_val"].argmax(1)), mb["name"][:26])
utils.print_metrics(val_metrics_ens, "ENSEMBLE")
print(f"\nGanancia ensemble vs large solo: {val_metrics_ens['f1_macro'] - f1_large:+.4f}")


Miembros: ['FacebookAI/xlm-roberta-large', 'FacebookAI/xlm-roberta-base', 'dccuchile/bert-base-spanish-wwm-cased']
Pesos ∝F1: [0.339, 0.326, 0.335]
[FacebookAI/xlm-roberta-lar] F1-macro         : 0.2941
[FacebookAI/xlm-roberta-lar] Accuracy         : 0.2980
[FacebookAI/xlm-roberta-lar] Mean dist (dec.) : 2.91
[FacebookAI/xlm-roberta-lar] <=1 decade       : 47.4%
[FacebookAI/xlm-roberta-lar] <=2 decades      : 63.5%
[FacebookAI/xlm-roberta-lar] Inter-century err: 21.6%
[FacebookAI/xlm-roberta-bas] F1-macro         : 0.2825
[FacebookAI/xlm-roberta-bas] Accuracy         : 0.2862
[FacebookAI/xlm-roberta-bas] Mean dist (dec.) : 2.97
[FacebookAI/xlm-roberta-bas] <=1 decade       : 48.4%
[FacebookAI/xlm-roberta-bas] <=2 decades      : 62.9%
[FacebookAI/xlm-roberta-bas] Inter-century err: 21.6%
[dccuchile/bert-base-spanis] F1-macro         : 0.2908
[dccuchile/bert-base-spanis] Accuracy         : 0.2900
[dccuchile/bert-base-spanis] Mean dist (dec.) : 3.02
[dccuchile/bert-base-spanis] <=1 decade

## 6. Predicción final + submission

In [8]:
ens_eval = dl_utils.combine_proba([mb["proba_eval"] for mb in members], W)
pred_decade = np.array([utils.IDX_TO_DECADE[i] for i in ens_eval.argmax(1)])

sub_path = utils.write_submission(corpus.eval_["id"].values, pred_decade, f"{OUT_DIR}/submission_11.csv")
utils.write_submission(corpus.eval_["id"].values, pred_decade, f"{OUT_DIR}/submission.csv")
print("Final submission ->", sub_path, "| décadas únicas:", len(set(pred_decade)))


Final submission -> /kaggle/working/submission_11.csv | décadas únicas: 39


## 7. Guardado del bundle (formato uniforme para NB08)

Los dirs de los modelos reusados conservan su nombre de NB07 → en NB08 se resuelven por basename
montando a la vez `modelo-final-dl-large` (este output) y `modelo-final-dl` (NB07).


In [9]:
payload = {
    "version": "11_xlmr_large_ensemble",
    "kind": "multitask_ensemble",
    "models": [{"name": mb["name"], "dir": mb["dir"], "use_century": mb["use_century"],
                "val_f1": float(mb["val_f1"]), "weight": float(W[i])}
               for i, mb in enumerate(members)],
    "max_length": MAX_LENGTH,
    "num_decades": utils.NUM_DECADES,
    "num_centuries": utils.NUM_CENTURIES,
    "idx_to_decade": utils.IDX_TO_DECADE,
    "val_f1_macro": float(val_metrics_ens["f1_macro"]),
    "val_metrics": val_metrics_ens,
    "multi_task": {"alpha_century": ALPHA_CENTURY},
    "regularization": {"dropout": DROPOUT, "loss_kind": LOSS_KIND, "smooth_eps": SMOOTH_EPS},
    "reused_from_nb07": [r["name"] for r in reused],
    "history": {"large": hist_large},
    "seed": utils.SEED,
}
joblib.dump(payload, f"{OUT_DIR}/modelo_final.joblib", compress=3)
print("Saved bundle:", f"{OUT_DIR}/modelo_final.joblib")


Saved bundle: /kaggle/working/modelo_final.joblib


### Descarga directa (FileLink) — evita el `Save Version` que mata la sesión

Para NB08 hay que publicar como Kaggle Dataset el `modelo_final.joblib` **junto con** la carpeta
`modelo_v11_A_xlmr_large/` (zip vía FileLink → New Dataset, preservando estructura).


In [10]:
from IPython.display import FileLink, display

for name in ("submission_11.csv", "submission.csv", "modelo_final.joblib"):
    path = f"{OUT_DIR}/{name}"
    if os.path.exists(path):
        print(f"{name} ({os.path.getsize(path)/1e6:.1f} MB)")
        display(FileLink(path))
print("\nNo olvides incluir la carpeta del modelo grande en el dataset:")
print(" ", MODEL_LARGE_DIR)


submission_11.csv (0.0 MB)


/kaggle/working/submission_11.csv

submission.csv (0.0 MB)


/kaggle/working/submission.csv

modelo_final.joblib (0.0 MB)


/kaggle/working/modelo_final.joblib


No olvides incluir la carpeta del modelo grande en el dataset:
  /kaggle/working/modelo_v11_A_xlmr_large


## 8. Iteraciones (rúbrica §5.2.1)

| # | Cambio principal | F1 holdout | Kaggle |
|---|---|---|---|
| Legacy E1 | TF-IDF char_wb+word + stacking | 0.275 (val) | 0.296 |
| 09 | XLM-R-base limpio (CE plana) | 0.288 | 0.29130 |
| 07 | ensemble DL base (XLM-R + BETO) | 0.3085 | 0.30276 |
| 08 | híbrido legacy + DL (barrido w) | 0.3305 | 0.31805 |
| **11** | **xlm-roberta-large + reuso BETO/XLM-R-base** | **REPORTAR** | **REPORTAR** |

**Hipótesis:** el techo es el encoder base; large (mismo arch que el base que rinde, +capacidad)
debería superar el 0.3085 del ensemble base-size en solitario, y el ensemble con los modelos
reusados de NB07 debería superar a large solo. Luego NB08 le suma el legacy (híbrido) → submission final.

**Decisiones:** reuso de NB07 (cero GPU desperdiciada en reentrenar BETO/base); split canónico
(holdout limpio, sin fuga); gradient checkpointing (cabe en T4); CE plana + multi-task siglo (α=0.10).
**Próximo:** NB08 con este bundle (legacy cacheado, ~10 min) y, fase 2, EuroBERT como miembro de diversidad.
